# Rossmann Sales - Thực nghiệm mô hình

Notebook chạy đúng bộ benchmark trong `scripts/experiments/rossmann_4_experiments.py`:

1. Baseline với dữ liệu gốc và ordinal encoding.
2. So sánh target encoding và native categorical cho LightGBM.
3. Feature engineering nâng cao, tham số tuning và early stopping.
4. Tối ưu hệ số post-processing để giảm RMSPE.

Mỗi thực nghiệm so sánh 5 mô hình: Dummy Median, Ridge, Decision Tree, Random Forest và LightGBM.

In [ ]:
from pathlib import Path
import importlib.util
import sys

cwd = Path.cwd().resolve()
root = next((p for p in [cwd, *cwd.parents] if (p / "scripts" / "experiments" / "rossmann_4_experiments.py").is_file()), None)
if root is None:
    raise RuntimeError("Không tìm thấy thư mục gốc dự án")

sys.path.insert(0, str(root))

script_path = root / "scripts" / "experiments" / "rossmann_4_experiments.py"
spec = importlib.util.spec_from_file_location("rossmann_experiments", script_path)
experiments = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiments)

print("Project root:", root)

## Chuẩn bị dữ liệu và chia holdout

Hàm chuẩn bị:
- nối `train.csv` với `store.csv`;
- tính `IsOpenYesterday` và `IsOpenTomorrow` trước khi lọc;
- loại các dòng `Open != 1` và `Sales <= 0` khỏi tập huấn luyện benchmark;
- tạo đặc trưng thời gian và đặc trưng mở cửa của đối thủ/khuyến mãi;
- giữ 42 ngày cuối làm validation theo thời gian.

In [ ]:
train_data, val_data = experiments.load_and_prepare_data(str(experiments.DATA_DIR))

print("Train:", len(train_data), "rows")
print("Validation:", len(val_data), "rows")
print("Validation dates:", val_data["Date"].min().date(), "->", val_data["Date"].max().date())

## Chạy toàn bộ 4 thực nghiệm

Tập dự đoán của từng mô hình huấn luyện được biến đổi ngược `expm1` từ target `log1p(Sales)`. Điểm đánh giá là RMSPE trên validation, nên chỉ bao gồm các dòng có doanh số dương.

In [ ]:
results = experiments.run_all_experiments(train_data, val_data)

results_csv = root / "results" / "rossmann_4_experiments_metrics.csv"
results.to_csv(results_csv, index=False)

experiments.plot_4x5_comparison(results)
display(results)

## Ma trận RMSPE

Bảng dưới so sánh kết quả 4 thực nghiệm x 5 mô hình. Giá trị thấp hơn là tốt hơn.

In [ ]:
import pandas as pd

pivot = results.pivot(index="Mô hình", columns="Thực nghiệm", values="RMSPE")
pivot = pivot.sort_values("TN4: Post-processing (RMSPE)", ascending=False)
display(pivot.style.format("{:.5f}"))

## Kết luận thực nghiệm

- TN1 là baseline benchmark trên dữ liệu gốc.
- TN2 kiểm tra cách mã hóa categorical: các mô hình sklearn dùng target encoding, riêng LightGBM dùng native category.
- TN3 bổ sung đặc trưng nâng cao và tuning.
- TN4 áp dụng hệ số nhân alpha cho dự đoán TN3 để tối ưu RMSPE trên cùng holdout.

Vì TN4 tối ưu trực tiếp trên tập validation, kết quả TN4 không được dùng như đánh giá độc lập chưa từng nhìn. CSV và biểu đồ trong `results/` là kết quả benchmark thực nghiệm này.

Trong triển khai production, mô hình chính được huấn luyện và đánh giá riêng tại `backend/app/model/train_model.py`, lưu kết quả vào `results/evaluation_metrics.json` và mô hình vào `models/rossmann_sales_lgbm.joblib`.

In [ ]:
final = results.loc[
    results["Thực nghiệm"] == "TN4: Post-processing (RMSPE)",
    ["Mô hình", "RMSPE", "Train Time (s)", "Inference (ms)", "Size (MB)"],
]
final = final.sort_values("RMSPE")

display(final.style.format({
    "RMSPE": "{:.5f}",
    "Train Time (s)": "{:.2f}",
    "Inference (ms)": "{:.2f}",
    "Size (MB)": "{:.3f}",
}))